# Standard LDA Training on PROMISE Data

This notebook trains a standard LDA model on the PROMISE dataset, following a similar preprocessing and evaluation pipeline as the Seeded LDA code for comparison. The key difference is the absence of seed word guidance (no `eta` matrix). The code processes the `PROMISE_exp_cleaned.csv` dataset, applies preprocessing, detects bigrams, trains an LDA model, and evaluates it with coherence scores.

## Import Libraries and Configure Logging

In [27]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phrases, Phraser
from tqdm import tqdm
import logging
import random
import os
from scipy.stats import entropy

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.getLogger('gensim').setLevel(logging.WARNING)

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

## Initialize NLP Tools and Stopwords

In [28]:
# Initialize NLP Tools and Stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Domain-specific and review-specific stopwords (same as Seeded LDA)
custom_stopwords = {
    'allow', 'class', 'available', 'part', 'case', 'lead', 'shall', 'product', 'operate',
    'operational', 'result', 'input', 'dependent', 'preference', 'item', 'without', 'let',
    'returned', 'message', 'every', 'system', 'run', 'fully', 'major', 'reasonable',
    'software', 'user', 'able', 'ability', 'support', 'year', 'expected', 'must',
    'information', 'data', 'use', 'using', 'provide', 'successfully', 'one', 'waiter',
    'include', 'accommodate', 'event', 'technique', 'recent', 'administrator', 'search',
    'add', 'allows', 'achieve', 'way', 'outside', 'release', 'launch', 'allowed', 'entered',
    'within', 'first', 'new', 'izogn', 'wcs', 'course', 'time', 'help', 'learn', 'ccr', 'cma',
    'review', 'star', 'rating', 'good', 'great', 'course', 'learn', 'learning',
    'would', 'like', 'could', 'one', 'bit', 'week', 'think', 'much', 'really',
    'lot', 'new', 'thank', 'thanks', 'many', 'well', 'also', 'get', 'time',
    'truly', 'even', 'make', 'see', 'content', 'material', 'class', 'work',
    'way', 'understand', 'information', 'helpful', 'useful', 'knowledge',
    'day', 'help', 'easy'
}
stop_words.update(custom_stopwords)

## Load PROMISE Data

In [29]:
def load_promise_data(file_path):
    try:
        df = pd.read_csv(file_path)
        logging.info(f"Successfully loaded PROMISE data from {file_path}")
        return df
    except FileNotFoundError:
        logging.error(f"PROMISE data file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading PROMISE data: {e}")
        raise

promise_data_path = '../../datasets/PROMISE_exp_cleaned.csv'
promise_df = load_promise_data(promise_data_path)

2025-07-10 10:28:49,089 - INFO - Successfully loaded PROMISE data from ../../datasets/PROMISE_exp_cleaned.csv


## Aggregate Data by Category

In [30]:
logging.info("Started aggregating PROMISE data by category")
category_texts = promise_df.groupby('_class_')['cleaned_text'].apply(lambda x: ' '.join(x)).to_dict()

for category, text in category_texts.items():
    word_count = len(text.split())
    logging.info(f"Category {category}: {word_count} words")
    print(f"Category {category}: {word_count} words")
    if word_count < 50:
        logging.warning(f"Category {category} has short text")
        print(f"Warning: Category {category} has short text ({word_count} words). Consider reviewing data.")
    if word_count == 0:
        logging.error(f"Category {category} has empty text")
        raise ValueError(f"Category {category} has empty text. Check data filtering.")

2025-07-10 10:28:49,098 - INFO - Started aggregating PROMISE data by category
2025-07-10 10:28:49,102 - INFO - Category F: 3041 words
2025-07-10 10:28:49,102 - INFO - Category FT: 129 words
2025-07-10 10:28:49,102 - INFO - Category PE: 559 words
2025-07-10 10:28:49,102 - INFO - Category PO: 58 words
2025-07-10 10:28:49,102 - INFO - Category SC: 144 words
2025-07-10 10:28:49,102 - INFO - Category SE: 951 words
2025-07-10 10:28:49,108 - INFO - Category US: 653 words


Category F: 3041 words
Category FT: 129 words
Category PE: 559 words
Category PO: 58 words
Category SC: 144 words
Category SE: 951 words
Category US: 653 words


## Load and Preprocess Review Data

In [31]:
input_path = '../../datasets/review_train.csv'
output_path = '../../datasets/standard_lda_reviews_with_topic.csv'
threshold = 0.8
entropy_threshold = 1.0  # For filtering noisy samples

os.makedirs('models', exist_ok=True)

logging.info("Started loading review data")
df = pd.read_csv(input_path)
df = df[['processed_reviews']].copy()
df['processed_reviews'] = df['processed_reviews'].fillna("")
logging.info("Completed loading review data")

2025-07-10 10:28:49,116 - INFO - Started loading review data
2025-07-10 10:28:49,495 - INFO - Completed loading review data


In [32]:
def preprocess(text):
    if not isinstance(text, str) or not text.strip():
        return []
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens
              if word not in stop_words and len(word) > 2]
    return tokens

logging.info("Started preprocessing reviews")
tqdm.pandas()
tokenized_reviews = df['processed_reviews'].progress_apply(preprocess)
valid_indices = [i for i, tokens in enumerate(tokenized_reviews) if tokens]
tokenized_reviews = [tokens for tokens in tokenized_reviews if tokens]
if not tokenized_reviews:
    raise ValueError("No valid reviews after preprocessing.")
logging.info("Completed preprocessing reviews")
filtered_df = df.iloc[valid_indices].copy()
logging.info(f"Filtered DataFrame to {len(filtered_df)} valid reviews")

2025-07-10 10:28:49,507 - INFO - Started preprocessing reviews
100%|██████████| 286518/286518 [00:09<00:00, 31570.24it/s]
2025-07-10 10:28:58,679 - INFO - Completed preprocessing reviews
2025-07-10 10:28:58,709 - INFO - Filtered DataFrame to 281527 valid reviews


## Detect Bigrams

In [33]:
logging.info("Started detecting bigrams")
sample_size = min(100000, len(tokenized_reviews))
sampled_reviews = random.sample(tokenized_reviews, sample_size) if sample_size < len(tokenized_reviews) else tokenized_reviews
bigram_model = Phrases(sampled_reviews, min_count=15, threshold=0.7, scoring='npmi')
bigram_phraser = Phraser(bigram_model)

logging.info("Started applying bigrams")
tokenized_reviews = [bigram_phraser[tokens] for tokens in tokenized_reviews]
logging.info("Completed applying bigrams")

2025-07-10 10:28:58,722 - INFO - Started detecting bigrams
2025-07-10 10:29:00,432 - INFO - Started applying bigrams
2025-07-10 10:29:02,511 - INFO - Completed applying bigrams


## Create Dictionary and Corpus

In [34]:
logging.info("Started creating dictionary")
dictionary = corpora.Dictionary(tokenized_reviews)
dictionary.filter_extremes(no_below=1, no_above=0.8)
logging.info("Completed creating dictionary")

2025-07-10 10:29:02,521 - INFO - Started creating dictionary
2025-07-10 10:29:04,930 - INFO - Completed creating dictionary


# Load Seed Words from Seeded LDA

In [35]:
def load_seed_words_from_csv(file_path):
    try:
        df = pd.read_csv(file_path)
        seed_words = df.groupby('Category')['SeedWord'].apply(list).to_dict()
        logging.info(f"Successfully loaded seed words from {file_path}")
        return seed_words
    except FileNotFoundError:
        logging.error(f"Seed words file not found at {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading seed words: {e}")
        raise

seed_words_path = '../../datasets/seed_words.csv'
seed_words = load_seed_words_from_csv(seed_words_path)

logging.info("Started creating BoW corpus")
corpus = [dictionary.doc2bow(text) for text in tokenized_reviews]
logging.info("Completed creating BoW corpus")

2025-07-10 10:29:04,942 - INFO - Successfully loaded seed words from ../../datasets/seed_words.csv
2025-07-10 10:29:04,942 - INFO - Started creating BoW corpus
2025-07-10 10:29:06,432 - INFO - Completed creating BoW corpus


## Train Standard LDA Model

In [36]:
num_topics = 7
logging.info("Started training LDA model")

lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    passes=20,
    alpha=0.01/num_topics,
    iterations=400,
    random_state=42,
    minimum_probability=0.05,
    per_word_topics=True,
    decay=0.7,
    offset=50.0
)

logging.info("Completed training LDA model")

2025-07-10 10:29:06,442 - INFO - Started training LDA model
2025-07-10 10:42:40,936 - INFO - Completed training LDA model


## Save Models

In [37]:
logging.info("Saving LDA model, dictionary, and bigram model")
lda_model.save('models/standard_lda_model')
dictionary.save('models/standard_dictionary')
bigram_phraser.save('models/standard_bigram_phraser')
logging.info("Saved LDA model, dictionary, and bigram models")

2025-07-10 10:42:40,942 - INFO - Saving LDA model, dictionary, and bigram model
2025-07-10 10:42:40,969 - INFO - Saved LDA model, dictionary, and bigram models


# Verify Seed Word Probabilities (Detailed)

In [46]:
logging.info("Checking Seed Word Probabilities and Ranks in Standard LDA Model")
print("\n--- Checking Seed Word Probabilities and Ranks in Standard LDA Model ---")
id_to_topic_name = {i: f"Topic_{i}" for i in range(num_topics)}

for topic_name, words in seed_words.items():
    print(f"\nSeed Words: {topic_name}")
    
    for word in words:
        if word in dictionary.token2id:
            word_id = dictionary.token2id[word]
            term_topics = lda_model.get_term_topics(word_id, minimum_probability=0.0)
            print(f"  - Seed Word '{word}'")
            for t_id, prob in term_topics:
                t_name = id_to_topic_name[t_id]
                topic_words_probs = lda_model.show_topic(t_id, topn=len(dictionary))
                word_to_rank = {w: idx + 1 for idx, (w, _) in enumerate(topic_words_probs)}
                rank = word_to_rank.get(word, "N/A")
                log_message = f"    * Topic {t_name} (ID: {t_id}): Probability = {prob:.4f}, Rank = {rank}"
                print(log_message)
        else:
            
            print(f"  - '{word}' (Not in dictionary)")
print("----------------------------------------------------------\n")

2025-07-10 10:48:19,091 - INFO - Checking Seed Word Probabilities and Ranks in Standard LDA Model



--- Checking Seed Word Probabilities and Ranks in Standard LDA Model ---

Seed Words: F
  - Seed Word 'player'
    * Topic Topic_3 (ID: 3): Probability = 0.0000, Rank = 1578
  - Seed Word 'display'
    * Topic Topic_5 (ID: 5): Probability = 0.0002, Rank = 414
  - Seed Word 'meeting'
    * Topic Topic_0 (ID: 0): Probability = 0.0001, Rank = 668
    * Topic Topic_1 (ID: 1): Probability = 0.0001, Rank = 848
    * Topic Topic_3 (ID: 3): Probability = 0.0000, Rank = 1945
  - Seed Word 'dispute'
    * Topic Topic_1 (ID: 1): Probability = 0.0001, Rank = 992
  - Seed Word 'program'
    * Topic Topic_0 (ID: 0): Probability = 0.0043, Rank = 50
    * Topic Topic_2 (ID: 2): Probability = 0.0009, Rank = 184
    * Topic Topic_3 (ID: 3): Probability = 0.0009, Rank = 283
    * Topic Topic_4 (ID: 4): Probability = 0.0007, Rank = 329
    * Topic Topic_5 (ID: 5): Probability = 0.0015, Rank = 127
  - Seed Word 'clinical'
    * Topic Topic_4 (ID: 4): Probability = 0.0015, Rank = 156
  - Seed Word 'member'

## Analyze Topics and Coherence

In [39]:
logging.info("Computing per-topic coherence scores")

coherence_model = CoherenceModel(
    model=lda_model,
    texts=tokenized_reviews,
    dictionary=dictionary,
    coherence='c_v',
    topn=10,
    window_size=50
)

per_topic_coherence = coherence_model.get_coherence_per_topic()
for i, score in enumerate(per_topic_coherence):
    logging.info(f"Topic #{i}: C_v Score = {score:.4f}")
print("------------------------------------\n")

2025-07-10 10:42:53,741 - INFO - Computing per-topic coherence scores
2025-07-10 10:43:12,249 - INFO - Topic #0: C_v Score = 0.4710
2025-07-10 10:43:12,250 - INFO - Topic #1: C_v Score = 0.4923
2025-07-10 10:43:12,250 - INFO - Topic #2: C_v Score = 0.5017
2025-07-10 10:43:12,251 - INFO - Topic #3: C_v Score = 0.7331
2025-07-10 10:43:12,251 - INFO - Topic #4: C_v Score = 0.5259
2025-07-10 10:43:12,253 - INFO - Topic #5: C_v Score = 0.4035
2025-07-10 10:43:12,254 - INFO - Topic #6: C_v Score = 0.3296


------------------------------------



In [40]:
logging.info("Discovered Topics (Top 30 words):")
print("\n--- Discovered Topics (Top 30 words) ---")

for i in range(num_topics):
    topic_words_probs = lda_model.show_topic(i, topn=30)
    topic_words = [word for word, prob in topic_words_probs]
    log_message = f"Topic #{i}: {', '.join(topic_words)}"
    logging.info(log_message)
print("----------------------------------------------------------\n")

2025-07-10 10:43:12,266 - INFO - Discovered Topics (Top 30 words):
2025-07-10 10:43:12,273 - INFO - Topic #0: best, python, learned, awesome, ever, language, know, amazing, experience, taken, next, teacher, chuck, looking_forward, specialization, feel, teaching, start, take, far, never, want, taking, already, love, sir, instructor, glad, got, fun
2025-07-10 10:43:12,275 - INFO - Topic #1: amazing, life, professor, love, wonderful, learned, opportunity, interesting, university, take, experience, everyone, team, excellent, psychology, recommend, better, people, made, future, world, put, happy, improve, positive, mind, study, making, daily, history
2025-07-10 10:43:12,277 - INFO - Topic #2: excellent, recommend, highly, clear, anyone, teaching, instructor, follow, professor, simple, interesting, fun, beginner, explanation, teacher, informative, interested, engaging, level, subject, made, taught, clearly, definitely, python, everything, easily, understandable, everyone, manner
2025-07-10 1


--- Discovered Topics (Top 30 words) ---
----------------------------------------------------------



## Compute Coherence Scores for Multiple topn Values

In [48]:
def compute_coherence_range(lda_model, tokenized_reviews, dictionary, window_size=50, topn_range=(10, 100, 10)):
    logging.info("Started computing C_v coherence scores for multiple topn values")
    print("\n--- Computing C_v Coherence for Multiple topn Values ---")
    start, end, step = topn_range
    coherence_results = []
    
    for topn in range(start, end + 1, step):
        print(f"\nComputing C_v Coherence for topn={topn}")
        
        coherence_model = CoherenceModel(
            model=lda_model,
            texts=tokenized_reviews,
            dictionary=dictionary,
            coherence='c_v',
            topn=topn,
            window_size=window_size
        )
        
        per_topic_coherence = coherence_model.get_coherence_per_topic()
        print(f"\nPer-Topic C_v Scores (topn={topn}):")
        for i, score in enumerate(per_topic_coherence):
            print(f"Topic #{i} - C_v Score = {score:.4f}")
        
        coherence_score = coherence_model.get_coherence()
        print(f"\nOverall C_v Coherence Score (topn={topn}): {coherence_score:.4f}")
        print("-" * 35)
        
        coherence_results.append({
            'topn': topn,
            'per_topic_coherence': per_topic_coherence,
            'overall_coherence': coherence_score
        })
    
    logging.info("Completed computing C_v coherence scores for multiple topn values")
    print("\n--- Completed Computing C_v Coherence for Multiple topn Values ---")
    return coherence_results

coherence_results = compute_coherence_range(
    lda_model=lda_model,
    tokenized_reviews=tokenized_reviews,
    dictionary=dictionary,
    window_size=50,
    topn_range=(10, 100, 10)
)

2025-07-10 10:49:22,301 - INFO - Started computing C_v coherence scores for multiple topn values



--- Computing C_v Coherence for Multiple topn Values ---

Computing C_v Coherence for topn=10

Per-Topic C_v Scores (topn=10):
Topic #0 - C_v Score = 0.4710
Topic #1 - C_v Score = 0.4923
Topic #2 - C_v Score = 0.5017
Topic #3 - C_v Score = 0.7331
Topic #4 - C_v Score = 0.5259
Topic #5 - C_v Score = 0.4035
Topic #6 - C_v Score = 0.3296

Overall C_v Coherence Score (topn=10): 0.4939
-----------------------------------

Computing C_v Coherence for topn=20

Per-Topic C_v Scores (topn=20):
Topic #0 - C_v Score = 0.4987
Topic #1 - C_v Score = 0.4649
Topic #2 - C_v Score = 0.4441
Topic #3 - C_v Score = 0.7605
Topic #4 - C_v Score = 0.4391
Topic #5 - C_v Score = 0.3882
Topic #6 - C_v Score = 0.3034

Overall C_v Coherence Score (topn=20): 0.4713
-----------------------------------

Computing C_v Coherence for topn=30

Per-Topic C_v Scores (topn=30):
Topic #0 - C_v Score = 0.5087
Topic #1 - C_v Score = 0.5043
Topic #2 - C_v Score = 0.4580
Topic #3 - C_v Score = 0.7854
Topic #4 - C_v Score = 0.5

2025-07-10 10:53:44,771 - INFO - Completed computing C_v coherence scores for multiple topn values



Overall C_v Coherence Score (topn=100): 0.5520
-----------------------------------

--- Completed Computing C_v Coherence for Multiple topn Values ---


Assign Topics to Reviews and Save to Single File

In [ ]:
logging.info("Started getting topic distributions")
doc_topics = [lda_model.get_document_topics(doc, minimum_probability=0.0) for doc in corpus]
topic_matrix = np.zeros((len(corpus), num_topics))

for i, topics in enumerate(doc_topics):
    for topic_id, prob in topics:
        topic_matrix[i, topic_id] = prob
        
logging.info("Completed getting topic distributions")
logging.info("Started saving all reviews with topics")

all_topics_data = []
entropies = [entropy(probs) for probs in topic_matrix]
logging.info(f"Average topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")
print(f"\nAverage topic entropy: {np.mean(entropies):.4f}, Std: {np.std(entropies):.4f}")

for i, (review_text, topic_probs) in enumerate(zip(filtered_df['processed_reviews'], topic_matrix)):
    dominant_topic = np.argmax(topic_probs)
    dominant_prob = topic_probs[dominant_topic]
    topic_name = f"Topic_{dominant_topic}"
    review_entropy = entropy(topic_probs)
    
    all_topics_data.append({
        'original_index': valid_indices[i],
        'processed_reviews': review_text,
        'topic': dominant_topic,
        'topic_name': topic_name,
        'confidence': dominant_prob,
        'topic_probs': ','.join(map(str, topic_probs)),
        'entropy': review_entropy
    })

all_topics_df = pd.DataFrame(all_topics_data)
all_topics_path = '../../datasets/standard_lda_reviews_with_topic.csv'
all_topics_df.to_csv(all_topics_path, index=False)
logging.info(f"All reviews with topic assignments saved to {all_topics_path}")
print(f"\nAll reviews with topic assignments saved to {all_topics_path}")

logging.info("Sample of saved data:")
logging.info(all_topics_df[['processed_reviews', 'topic', 'topic_name', 'confidence', 'entropy']].head(2).to_string())
print("\nSample of saved data:")
print(all_topics_df[['processed_reviews', 'topic', 'topic_name', 'confidence', 'entropy']].head(2))

2025-07-10 10:04:01,215 - INFO - Started getting topic distributions
2025-07-10 10:04:39,659 - INFO - Completed getting topic distributions
